# TrustMind AI — SWMH Preprocessing

**MSc Artificial Intelligence Dissertation**

This notebook applies the preprocessing plan from `01_SWMH_EDA.ipynb` and writes cleaned train / validation / test CSVs for downstream LLM-only and LLM+RAG experiments.

**Pipeline steps**
1. UTF-8 CSV load
2. HTML entity decoding
3. URL removal
4. Username anonymisation (`u/name` → `[USER]`)
5. Whitespace normalisation
6. Empty-row and duplicate removal
7. Head-preserving truncation to 512 words
8. Cross-split leakage removal (train posts that also appear in val/test)

Aggressive stemming / stop-word removal is **not** applied, to preserve emotionally meaningful vocabulary.


## Section 1 — Imports and Paths


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "research":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "research"))
from preprocessing import (
    clean_text,
    load_swmh_split,
    preprocess_dataframe,
    preprocess_swmh_directory,
)

DATA_DIR = ROOT / "datasets" / "synthetic_wellbeing"
OUT_DIR = DATA_DIR / "processed"
FIGURES_DIR = ROOT / "research" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "figure.figsize": (10, 5),
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("Project root:", ROOT)
print("Data dir:", DATA_DIR)
print("Output dir:", OUT_DIR)


## Section 2 — Before / After Cleaning Demo

Show how a raw post is transformed by the text cleaner.


In [ ]:
demo = (
    "I feel hopeless &amp;nbsp; check https://example.com "
    "and talk to u/someuser about this.\n\nReally overwhelmed."
)
print("RAW:")
print(demo)
print("\nCLEANED:")
print(clean_text(demo))


## Section 3 — Run Full Preprocessing Pipeline

Process all three splits and write cleaned CSVs under `datasets/synthetic_wellbeing/processed/`.


In [ ]:
stats = preprocess_swmh_directory(
    DATA_DIR,
    OUT_DIR,
    max_words=512,
    remove_cross_split_leakage=True,
)

for split in ("train", "validation", "test"):
    s = stats[split]
    print(f"\n=== {split.upper()} ===")
    print(f"Rows before:        {s['rows_before']:,}")
    print(f"Rows after:         {s['rows_after']:,}")
    print(f"Empty removed:      {s['empty_removed']:,}")
    print(f"Duplicates removed: {s['duplicates_removed']:,}")
    print(f"Truncated posts:    {s['truncated_posts']:,}")
    print(f"Mean word count:    {s['mean_word_count']:.1f}")
    print(f"Median word count:  {s['median_word_count']:.1f}")

if "leakage" in stats:
    leak = stats["leakage"]
    print("\n=== CROSS-SPLIT LEAKAGE ===")
    print(f"Train before:       {leak['train_before']:,}")
    print(f"Train after:        {leak['train_after']:,}")
    print(f"Leakage removed:    {leak['leakage_removed']:,}")

print("\nSaved files:")
for name, path in stats["output_files"].items():
    print(f"  {name}: {path}")


## Section 4 — Verify Cleaned Splits

Confirm schema, label coverage, and that no empty text remains.


In [ ]:
clean_paths = {
    "train": OUT_DIR / "train_clean.csv",
    "validation": OUT_DIR / "val_clean.csv",
    "test": OUT_DIR / "test_clean.csv",
}

frames = {}
for name, path in clean_paths.items():
    df = pd.read_csv(path, encoding="utf-8")
    frames[name] = df
    print(f"\n--- {name} ---")
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    print("missing text:", int(df["text"].isna().sum()))
    print("empty text:", int((df["text"].astype(str).str.strip() == "").sum()))
    print("labels:")
    print(df["label"].value_counts().sort_index())

combined = pd.concat(
    [df.assign(split=name) for name, df in frames.items()],
    ignore_index=True,
)
print("\nCombined cleaned rows:", f"{len(combined):,}")
print("Truncated overall:", f"{int(combined['truncated'].sum()):,} "
      f"({100 * combined['truncated'].mean():.2f}%)")


## Section 5 — Word-Count Distribution After Cleaning


In [ ]:
fig, ax = plt.subplots()
ax.hist(combined["word_count"], bins=60, color="#0d9488", edgecolor="white", alpha=0.85)
ax.axvline(combined["word_count"].median(), color="#dc2626", linestyle="--",
           label=f"Median = {combined['word_count'].median():.0f}")
ax.set_title("Word count after preprocessing")
ax.set_xlabel("Words")
ax.set_ylabel("Number of posts")
ax.legend()
fig.tight_layout()

out_path = FIGURES_DIR / "word_count_after_preprocessing.png"
fig.savefig(out_path)
plt.show()
print("Saved:", out_path)


## Section 6 — Link to Research Question

Preprocessing prepares **identical inputs** for both experimental conditions:

| Condition | Uses cleaned synthetic set? | Why identical preprocessing matters |
|-----------|--------------------|-------------------------------------|
| **LLM-only** | Yes | Isolates the effect of the model, not data cleaning |
| **LLM + RAG** | Yes | Same text + retrieved external sources |

Truncation and leakage removal reduce confounds so any later gains in trustworthiness, reliability, or explainability can be attributed to **RAG grounding**, not unclean data.

**Next step:** train / evaluate a baseline classifier (or LLM prompt baseline) on `datasets/synthetic_wellbeing/processed/*_clean.csv`, then add the RAG arm.
